In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from langdetect import detect
from textblob import TextBlob
from collections import Counter
import time, re, json, pandas as pd, logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --------------------------
# UTILITAIRES (ton code enrichi)
# --------------------------
def detect_langue(texte):
    try:
        lang = detect(texte)
        if lang == "ar": return "Arabe"
        if lang == "fr": return "Français"
        if lang == "en": return "Anglais"
        return "Darija"
    except:
        return "Inconnue"

def detect_polarite(texte):
    try:
        return round(TextBlob(texte).sentiment.polarity, 3)
    except:
        return 0

def detect_categorie(texte):
    t = texte.lower()
    if any(w in t for w in ["sport", "match", "football", "foot"]): return "Sport"
    if any(w in t for w in ["roi", "gouvernement", "politique", "ministre"]): return "Politique"
    if any(w in t for w in ["film", "artiste", "musique"]): return "Art"
    if any(w in t for w in ["économie", "finance", "entreprise"]): return "Économie"
    return "Autre"

def parse_number_from_text(s):
    """Return int from text like '1,234' or '1.2K' or '1.2 M'."""
    if not s:
        return 0
    s = s.strip()
    # handle K / M shorthand
    m = re.match(r'([\d\.,]+)\s*([KMkM]?)', s)
    if m:
        num = m.group(1).replace(',', '').replace('.', '')
        # if letter present, handle approx
        letter = m.group(2).upper()
        try:
            base = float(m.group(1).replace(',', ''))
        except:
            base = None
        if letter == 'K':
            return int(base * 1000) if base is not None else 0
        if letter == 'M':
            return int(base * 1_000_000) if base is not None else 0
        # no letter: plain int
        try:
            return int(float(m.group(1).replace(',', '')))
        except:
            return 0
    # fallback to digits
    m2 = re.search(r'(\d[\d,\. ]+)', s)
    if m2:
        return int(m2.group(1).replace(',', '').replace(' ', '').split('.')[0])
    return 0

# --------------------------
# EXTRACTION SPÉCIFIQUE DES TWEETS ET COMMENTAIRES
# --------------------------
def extract_tweet_data(tweet_element):
    """Extrait les données structurées d'un élément tweet (méthode précédente)"""
    try:
        tweet_data = {}
        
        # Auteur du tweet
        try:
            author = tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="User-Name"]')
            tweet_data['auteur'] = author.text.split('\n')[0]
            tweet_data['compte'] = author.find_element(By.CSS_SELECTOR, 'a[href*="/"]').get_attribute('href').split('/')[-1]
        except:
            tweet_data['auteur'] = "Non trouvé"
            tweet_data['compte'] = "Non trouvé"
        
        # Contenu du tweet
        try:
            content = tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="tweetText"]')
            tweet_data['contenu'] = content.text
        except:
            tweet_data['contenu'] = "Non trouvé"
        
        # Date et heure
        try:
            time_element = tweet_element.find_element(By.CSS_SELECTOR, 'time')
            tweet_data['date_publication'] = time_element.get_attribute('datetime')
            tweet_data['heure_affichage'] = time_element.text
        except:
            tweet_data['date_publication'] = "Non trouvé"
            tweet_data['heure_affichage'] = "Non trouvé"
        
        # Statistiques d'engagement
        engagement_selectors = {
            'reponses': '[data-testid="reply"]',
            'retweets': '[data-testid="retweet"]', 
            'likes': '[data-testid="like"]',
            'vues': '[data-testid="app-text-transition-container"]'
        }
        
        tweet_data['statistiques'] = {}
        for stat, selector in engagement_selectors.items():
            try:
                element = tweet_element.find_element(By.CSS_SELECTOR, selector)
                # Chercher le nombre dans les spans enfants
                spans = element.find_elements(By.TAG_NAME, 'span')
                count = "0"
                for span in spans:
                    text = span.text.strip()
                    if text and text.isdigit():
                        count = text
                        break
                tweet_data['statistiques'][stat] = count
            except:
                tweet_data['statistiques'][stat] = "0"
        
        # Médias (vidéos/images)
        try:
            media = tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="tweetPhoto"]')
            tweet_data['has_media'] = True
            # Durée vidéo si disponible
            try:
                video_duration = media.find_element(By.CSS_SELECTOR, 'span[dir="ltr"]')
                tweet_data['duree_video'] = video_duration.text
            except:
                tweet_data['duree_video'] = "Non spécifiée"
        except:
            tweet_data['has_media'] = False
            tweet_data['duree_video'] = "Aucune"
        
        # Hashtags
        try:
            hashtags = tweet_element.find_elements(By.CSS_SELECTOR, 'a[href*="/hashtag/"]')
            tweet_data['hashtags'] = [tag.text for tag in hashtags]
        except:
            tweet_data['hashtags'] = []
        
        return tweet_data
        
    except Exception as e:
        logging.error(f"Erreur lors de l'extraction du tweet: {e}")
        return None

def get_video_duration_js(driver):
    """
    Parcourt tous les <video> dans la page via JS et retourne la 1ère durée utile (>0).
    Retour en secondes (int) ou 0 si non trouvé.
    """
    try:
        script = """
        let vids = Array.from(document.querySelectorAll('video'));
        let durations = vids.map(v => v.duration || 0);
        return durations;
        """
        durations = driver.execute_script(script)
        if durations:
            for d in durations:
                try:
                    if d and d > 0 and d < 60*60*10:  # filtre des durées absurdes
                        return int(round(d))
                except:
                    continue
    except Exception as e:
        logging.warning(f"get_video_duration_js erreur: {e}")
    return 0

def get_views(driver):
    """Essaye plusieurs méthodes pour récupérer le nombre de vues."""
    # 1) Recherche d'éléments avec aria-label contenant 'Views' / 'Vues'
    try:
        candidates = driver.find_elements(By.XPATH, "//*[contains(@aria-label, 'Views') or contains(@aria-label, 'Vues') or contains(@aria-label, 'vue') or contains(@aria-label, 'views')]")
        for el in candidates:
            al = el.get_attribute("aria-label") or ""
            num = parse_number_from_text(al)
            if num > 0:
                logging.info(f"Views trouvé via aria-label: {num}")
                return num
    except Exception as e:
        logging.debug(f"views aria-label err: {e}")

    # 2) Recherche d'un span ou div qui contient 'Views'/'Vues' comme texte et prendre sibling num
    try:
        for label in ["Views", "Vues", "vues", "views"]:
            els = driver.find_elements(By.XPATH, f"//span[text()='{label}' or contains(text(), '{label}')]")
            for lab in els:
                # essayer sibling ou parent
                try:
                    parent = lab.find_element(By.XPATH, "./..")
                    # chercher chiffres dans le parent
                    text = parent.text
                    num = parse_number_from_text(text)
                    if num > 0:
                        logging.info(f"Views trouvé via label sibling: {num}")
                        return num
                except:
                    continue
    except Exception as e:
        logging.debug(f"views label err: {e}")

    # 3) Chercher motifs textes généraux sur la page (fallback)
    try:
        page_text = driver.find_element(By.TAG_NAME, "body").text
        # trouver motifs comme "15,000 views" ou "15k views" ou "15 000 vues"
        m = re.search(r'(\d[\d\.,\s]*\d)\s*(?:views|vues|Views|Vues|views)', page_text)
        if m:
            num = parse_number_from_text(m.group(1))
            if num > 0:
                logging.info(f"Views trouvé via page_text fallback: {num}")
                return num
    except Exception as e:
        logging.debug(f"views fallback err: {e}")

    logging.warning("Nombre de vues non trouvé.")
    return 0

def extract_metrics(driver):
    """Récupère Likes, Reposts (retweets), Replies (commentaires), Bookmarks si possible."""
    metrics = {"Replies": 0, "Reposts": 0, "Likes": 0, "Bookmarks": 0, "Views": 0}
    try:
        button_tests = {
            "Likes": "like",
            "Replies": "reply",
            "Bookmarks": "bookmark"
        }
        for metric, testid in button_tests.items():
            try:
                button = driver.find_element(By.XPATH, f"//button[@data-testid='{testid}']")
                # essayer aria-label puis texte
                aria_label = button.get_attribute("aria-label") or button.text or ""
                num_match = re.search(r'(\d[\d,\.kKmM ]*)', aria_label)
                if num_match:
                    metrics[metric] = parse_number_from_text(num_match.group(1))
                    logging.info(f"{metric} trouvé : {metrics[metric]}")
            except Exception as e:
                logging.debug(f"{metric} non trouvé: {e}")

        # Reposts / retweets
        for testid in ["retweet_or_repost", "retweet"]:
            try:
                button = driver.find_element(By.XPATH, f"//button[@data-testid='{testid}']")
                aria_label = button.get_attribute("aria-label") or button.text or ""
                num = parse_number_from_text(aria_label)
                if num > 0:
                    metrics["Reposts"] = num
                    break
            except:
                continue

        # Views via fonction dédiée
        metrics["Views"] = get_views(driver)
    except Exception as e:
        logging.error(f"Erreur générale métriques : {e}")
    logging.info(f"Métriques extraites: {metrics}")
    return metrics

def get_comments_texts(driver, max_comments=200):
    """
    Récupère les textes des replies en utilisant la méthode d'extraction structurée.
    Version améliorée avec extraction des données complètes des commentaires.
    """
    comments_data = []
    try:
        # Scroll progressif pour charger plus de replies
        for _ in range(8):
            driver.execute_script("window.scrollBy(0, 1200);")
            time.sleep(1.2)

        # Trouver tous les tweets (sélecteur principal)
        tweet_elements = driver.find_elements(By.CSS_SELECTOR, '[data-testid="tweet"]')
        logging.info(f"Tweets trouvés: {len(tweet_elements)}")
        
        if len(tweet_elements) <= 1:
            logging.warning("Peu de tweets trouvés — peut-être contenu pas chargé totalement.")
        
        # Skip main tweet (le premier tweet)
        for tweet_element in tweet_elements[1:]:
            if len(comments_data) >= max_comments:
                break
            try:
                # Utiliser la méthode d'extraction structurée
                comment_data = extract_tweet_data(tweet_element)
                if comment_data and comment_data.get('contenu') and comment_data['contenu'] != "Non trouvé":
                    # Ajouter l'analyse linguistique et sentimentale
                    comment_data['langue'] = detect_langue(comment_data['contenu'])
                    comment_data['polarite'] = detect_polarite(comment_data['contenu'])
                    comment_data['categorie'] = detect_categorie(comment_data['contenu'])
                    comments_data.append(comment_data)
            except Exception as e:
                logging.debug(f"Erreur extraction commentaire structuré: {e}")
                continue
                
    except Exception as e:
        logging.error(f"Erreur get_comments_texts: {e}")
    
    logging.info(f"Commentaires récupérés (structurés): {len(comments_data)}")
    return comments_data

# --------------------------
# ANALYSE DES COMMENTAIRES
# --------------------------
def analyze_comments(comments_data):
    """Analyse les commentaires avec données structurées"""
    analyzed = []
    langs = Counter()
    polarities = Counter()
    word_counter = Counter()
    authors_counter = Counter()

    for comment in comments_data:
        text = comment.get('contenu', '')
        if not text:
            continue
            
        # Utiliser les données déjà extraites ou recalculer
        lang = comment.get('langue', detect_langue(text))
        pol_val = comment.get('polarite', detect_polarite(text))
        
        if pol_val > 0.1:
            pol = "positive"
        elif pol_val < -0.1:
            pol = "negative"
        else:
            pol = "neutral"
            
        analyzed.append({
            "text": text, 
            "language": lang, 
            "polarity_value": pol_val, 
            "sentiment": pol,
            "auteur": comment.get('auteur', 'Inconnu'),
            "compte": comment.get('compte', 'Inconnu'),
            "date": comment.get('date_publication', 'Inconnue'),
            "statistiques": comment.get('statistiques', {})
        })
        
        langs[lang] += 1
        polarities[pol] += 1
        authors_counter[comment.get('auteur', 'Inconnu')] += 1
        
        # Compter les mots
        words = re.findall(r"\b\w+\b", text.lower())
        word_counter.update(words)

    total = len(analyzed) or 1
    lang_pct = {k: round(v/total*100, 2) for k,v in langs.items()}
    pol_pct = {k: round(v/total*100, 2) for k,v in polarities.items()}
    top_words = dict(word_counter.most_common(10))
    top_authors = dict(authors_counter.most_common(5))

    stats = {
        "languages": lang_pct, 
        "polarities": pol_pct, 
        "top_words": top_words,
        "top_authors": top_authors,
        "total_comments": total
    }
    return analyzed, stats

# --------------------------
# SCRAPER TWEET (principale)
# --------------------------
def scraper_tweet(url, save_csv=True):
    options = Options()
    # options.add_argument("--headless=new")  # décommente pour headless
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=fr")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

    try:
        logging.info("Ouverture de l'URL...")
        driver.get(url)
        wait = WebDriverWait(driver, 30)

        # Scroll initial pour charger contenu
        for i in range(6):
            driver.execute_script("window.scrollBy(0, 1200);")
            time.sleep(1.5)

        data = {}
        
        # EXTRAIRE LE TWEET PRINCIPAL AVEC LA MÉTHODE STRUCTURÉE
        try:
            main_tweet_element = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '[data-testid="tweet"]')))
            main_tweet_data = extract_tweet_data(main_tweet_element)
            
            if main_tweet_data:
                data["Titre"] = main_tweet_data.get('contenu', '')
                data["Auteur"] = main_tweet_data.get('auteur', '')
                data["Compte"] = main_tweet_data.get('compte', '')
                data["Date publication"] = main_tweet_data.get('date_publication', '')
                logging.info(f"Titre principal: {data['Titre'][:100]}...")
        except Exception as e:
            logging.error(f"Erreur extraction tweet principal: {e}")
            data["Titre"] = ""

        # CATEGORIE
        data["Catégorie"] = detect_categorie(data.get("Titre", ""))

        # DUREE (JS)
        dur = get_video_duration_js(driver)
        data["Durée"] = f"{dur}s" if dur else "Inconnue"

        # METRICS (likes, replies, retweets) et VIEWS
        metrics = extract_metrics(driver)
        data["Likes"] = metrics.get("Likes", 0)
        data["Retweets"] = metrics.get("Reposts", 0)
        data["Commentaires"] = metrics.get("Replies", 0)
        data["Nombre de vues"] = metrics.get("Views", 0)
        data["Nombre de partages"] = metrics.get("Reposts", 0)
        data["Nombre de commentaires"] = metrics.get("Replies", 0)

        # CONTENU DES COMMENTAIRES (VERSION STRUCTURÉE)
        comments_data = get_comments_texts(driver, max_comments=300)
        analyzed_comments, comment_stats = analyze_comments(comments_data)
        data["comments"] = analyzed_comments
        data["comment_stats"] = comment_stats

        # MOTS PLUS CITÉS (depuis titre + commentaires top words)
        title_words = re.findall(r"\b\w+\b", data.get("Titre", "").lower())
        overall_counter = Counter(title_words)
        overall_counter.update(comment_stats.get("top_words", {}))
        data["Mots plus cités"] = dict(overall_counter.most_common(10))

        # LANGUE / % LANGUES
        data["Langue"] = detect_langue(data.get("Titre", "")) if data.get("Titre") else "Inconnue"
        data["% Langues"] = comment_stats.get("languages", {data["Langue"]: 100})

        # POLARITE
        data["Polarité"] = detect_polarite(data.get("Titre", ""))
        if data["Polarité"] > 0.1:
            sent = "Positive"
        elif data["Polarité"] < -0.1:
            sent = "Négative"
        else:
            sent = "Neutre"
        data["% Polarité"] = {sent: 100}
        data["% Polarité Twitter"] = comment_stats.get("polarities", {sent: 100})

        data["Lien tweet"] = url

        # Sauvegarde JSON + CSV
        with open("video_hespress_x.json", "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        if save_csv:
            # pour CSV, convertir commentaires en texte résumé
            flat = {
                "Titre": data.get("Titre", ""),
                "Auteur": data.get("Auteur", ""),
                "Compte": data.get("Compte", ""),
                "Catégorie": data.get("Catégorie", ""),
                "Date publication": data.get("Date publication", ""),
                "Durée": data.get("Durée", ""),
                "Likes": data.get("Likes", 0),
                "Retweets": data.get("Retweets", 0),
                "Nombre de vues": data.get("Nombre de vues", 0),
                "Nombre de partages": data.get("Nombre de partages", 0),
                "Nombre de commentaires": data.get("Nombre de commentaires", 0),
                "Mots plus cités": json.dumps(data.get("Mots plus cités", {}), ensure_ascii=False),
                "Langue": data.get("Langue", ""),
                "% Langues": json.dumps(data.get("% Langues", {}), ensure_ascii=False),
                "Polarité": data.get("Polarité", 0),
                "% Polarité": json.dumps(data.get("% Polarité", {}), ensure_ascii=False),
                "Lien tweet": data.get("Lien tweet", "")
            }
            df = pd.DataFrame([flat])
            df.to_csv("video_hespress_x.csv", index=False, encoding="utf-8-sig")

        print(json.dumps(data, indent=2, ensure_ascii=False))
        logging.info("Données sauvegardées.")
        return data

    except Exception as e:
        logging.error(f"Erreur générale scraper: {e}")
        return {}
    finally:
        driver.quit()

# --------------------------
# EXEMPLE D'EXECUTION
# --------------------------
if __name__ == "__main__":
    url = "https://x.com/hespress/status/1980326222497460307"  # Remplace par l'URL réelle
    scraper_tweet(url)

2025-10-23 21:32:22,012 - INFO - ====== WebDriver manager ======
2025-10-23 21:32:29,274 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-23 21:32:29,407 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-23 21:32:29,534 - INFO - Driver [C:\Users\dell\.wdm\drivers\chromedriver\win64\141.0.7390.122\chromedriver.exe] found in cache
2025-10-23 21:32:31,517 - INFO - Ouverture de l'URL...
2025-10-23 21:32:46,643 - INFO - Titre principal: مغاربة يحتفلون بتتويج الأشبال عند الحدود مع الجزائر

#المغرب #الجزائر #أشبال_الأطلس #كرة_القدم #vira...
2025-10-23 21:32:46,927 - INFO - Likes trouvé : 66
2025-10-23 21:32:46,983 - INFO - Replies trouvé : 23
2025-10-23 21:32:47,032 - INFO - Bookmarks trouvé : 1
2025-10-23 21:32:47,151 - INFO - Views trouvé via aria-label: 23
2025-10-23 21:32:47,151 - INFO - Métriques extraites: {'Replies': 23, 'Reposts': 6, 'Likes': 66, 'Bookmarks': 1, 'Views': 23}
2025-10-23 21:32:56,984 - INFO - Tweets trouvés: 1
2025-10-23 21:32:56,984 -

{
  "Titre": "مغاربة يحتفلون بتتويج الأشبال عند الحدود مع الجزائر\n\n#المغرب #الجزائر #أشبال_الأطلس #كرة_القدم #viral #MH",
  "Auteur": "Hespress هسبريس",
  "Compte": "hespress",
  "Date publication": "2025-10-20T17:32:18.000Z",
  "Catégorie": "Autre",
  "Durée": "29s",
  "Likes": 66,
  "Retweets": 6,
  "Commentaires": 23,
  "Nombre de vues": 23,
  "Nombre de partages": 6,
  "Nombre de commentaires": 23,
  "comments": [],
  "comment_stats": {
    "languages": {},
    "polarities": {},
    "top_words": {},
    "top_authors": {},
    "total_comments": 1
  },
  "Mots plus cités": {
    "الجزائر": 2,
    "مغاربة": 1,
    "يحتفلون": 1,
    "بتتويج": 1,
    "الأشبال": 1,
    "عند": 1,
    "الحدود": 1,
    "مع": 1,
    "المغرب": 1,
    "أشبال_الأطلس": 1
  },
  "Langue": "Arabe",
  "% Langues": {},
  "Polarité": 0.0,
  "% Polarité": {
    "Neutre": 100
  },
  "% Polarité Twitter": {},
  "Lien tweet": "https://x.com/hespress/status/1980326222497460307"
}


In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time, re, json, pandas as pd, logging
import os
from urllib.parse import quote, unquote
from datetime import datetime

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --------------------------
# CONFIGURATION DU DRIVER
# --------------------------
def setup_driver():
    """Configure le driver Chrome"""
    options = Options()
    
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=fr")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2
    }
    options.add_experimental_option("prefs", prefs)
    
    try:
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        return driver
    except Exception as e:
        logging.error(f"Erreur création driver: {e}")
        return None

# --------------------------
# FONCTIONS DE SCRAPING GOOGLE AMÉLIORÉES
# --------------------------
def search_google_videos_2025(driver, query, max_results=100):
    """Recherche des vidéos sur Google pour septembre 2025 - VERSION AMÉLIORÉE"""
    try:
        # Encoder la requête pour l'URL
        encoded_query = quote(query)
        url = f"https://www.google.com/search?q={encoded_query}&tbm=vid&num=100"
        
        logging.info(f"🔍 Recherche Google 2025: {query}")
        driver.get(url)
        time.sleep(4)
        
        # Accepter les cookies si nécessaire
        try:
            accept_button = driver.find_element(By.XPATH, "//button[contains(., 'Tout accepter') or contains(., 'Accept all') or contains(., 'Accepter tout')]")
            accept_button.click()
            time.sleep(2)
        except:
            pass
        
        # Scroll agressif pour charger plus de résultats
        logging.info("📜 Scroll agressif pour charger plus de résultats...")
        for i in range(10):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1.5)
        
        # Essayer de cliquer sur "Plus de résultats" si disponible
        try:
            more_results = driver.find_elements(By.XPATH, "//a[contains(., 'Plus de résultats') or contains(., 'More results')]")
            if more_results:
                more_results[0].click()
                time.sleep(3)
                # Rescroll après avoir cliqué
                for i in range(5):
                    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                    time.sleep(1)
        except:
            pass
        
        return extract_video_links_improved(driver, max_results)
        
    except Exception as e:
        logging.error(f"❌ Erreur recherche Google 2025: {e}")
        return []

def extract_video_links_improved(driver, max_results):
    """Extrait les liens vidéo des résultats Google - VERSION AMÉLIORÉE"""
    video_links = []
    
    try:
        # Méthode 1: Chercher dans tous les liens de la page
        all_links = driver.find_elements(By.TAG_NAME, "a")
        
        for link in all_links:
            if len(video_links) >= max_results:
                break
                
            try:
                href = link.get_attribute("href")
                if href and ("x.com" in href or "twitter.com" in href):
                    # Nettoyer l'URL
                    clean_url = clean_twitter_url(href)
                    if clean_url and clean_url not in video_links:
                        video_links.append(clean_url)
                        logging.info(f"✅ Lien trouvé: {clean_url}")
            except:
                continue
        
        # Méthode 2: Chercher spécifiquement dans les résultats vidéo
        if len(video_links) < max_results:
            try:
                video_elements = driver.find_elements(By.CSS_SELECTOR, "div.g, div.video-result, div[data-ved], div[role='heading']")
                
                for element in video_elements:
                    if len(video_links) >= max_results:
                        break
                        
                    try:
                        link_selectors = ["a[href]", "a"]
                        for selector in link_selectors:
                            try:
                                links = element.find_elements(By.CSS_SELECTOR, selector)
                                for link in links:
                                    href = link.get_attribute("href")
                                    if href and ("x.com" in href or "twitter.com" in href):
                                        clean_url = clean_twitter_url(href)
                                        if clean_url and clean_url not in video_links:
                                            video_links.append(clean_url)
                                            logging.info(f"✅ Lien trouvé (méthode 2): {clean_url}")
                                            break
                            except:
                                continue
                    except:
                        continue
            except:
                pass
        
        # Méthode 3: Chercher par texte dans les liens
        if len(video_links) < max_results:
            try:
                twitter_links = driver.find_elements(By.XPATH, "//a[contains(@href, 'x.com') or contains(@href, 'twitter.com')]")
                for link in twitter_links:
                    if len(video_links) >= max_results:
                        break
                    href = link.get_attribute("href")
                    clean_url = clean_twitter_url(href)
                    if clean_url and clean_url not in video_links:
                        video_links.append(clean_url)
                        logging.info(f"✅ Lien trouvé (méthode 3): {clean_url}")
            except:
                pass
                
    except Exception as e:
        logging.error(f"❌ Erreur extraction liens vidéo: {e}")
    
    logging.info(f"📊 Total liens extraits: {len(video_links)}")
    return video_links

def clean_twitter_url(url):
    """Nettoie et valide les URLs Twitter"""
    try:
        # Si c'est une URL Google redirect, extraire le vrai URL
        if "google.com/url" in url:
            match = re.search(r'url=([^&]+)', url)
            if match:
                url = unquote(match.group(1))
        
        # Garder seulement les URLs Twitter/X
        if "x.com/" in url or "twitter.com/" in url:
            # S'assurer que c'est un lien de statut
            if "/status/" in url:
                # Prendre seulement la partie avant les paramètres
                clean_url = url.split('?')[0]
                return clean_url
                
    except Exception as e:
        logging.debug(f"Erreur nettoyage URL: {e}")
    
    return None

def extract_tweet_data_quick(driver):
    """Extrait rapidement les données du tweet (optimisé pour la vitesse)"""
    tweet_data = {
        "title": "",
        "timestamp": "",
        "author": "Hespress",
        "metrics": {"likes": 0, "retweets": 0}
    }
    
    try:
        # Titre du tweet - méthode rapide
        try:
            title_elements = driver.find_elements(By.CSS_SELECTOR, '[data-testid="tweetText"], article div[dir="auto"]')
            if title_elements:
                tweet_data["title"] = title_elements[0].text[:500]  # Limiter la longueur
        except:
            pass
        
        # Date - méthode rapide
        try:
            time_elements = driver.find_elements(By.TAG_NAME, "time")
            if time_elements:
                tweet_data["timestamp"] = time_elements[0].get_attribute("datetime")
        except:
            pass
            
    except Exception as e:
        logging.debug(f"Erreur extraction rapide données tweet: {e}")
    
    return tweet_data

# --------------------------
# SCRAPER PRINCIPAL OPTIMISÉ
# --------------------------
def scrape_hespress_videos_september_2025_max():
    """Scrape le MAXIMUM de vidéos Hespress de septembre 2025"""
    
    driver = setup_driver()
    if not driver:
        logging.error("❌ Impossible de créer le driver Chrome")
        return
    
    try:
        # REQUÊTES ÉTENDUES POUR MAXIMISER LES RÉSULTATS
        queries = [
            "hespress video twitter september 2025",
            "hespress x.com video septembre 2025", 
            "hespress twitter video month september 2025",
            "site:x.com hespress video september 2025",
            "hespress vidéo twitter septembre 2025",
            '"hespress" "video" "september 2025" site:x.com',
            'hespress "septembre 2025" video twitter',
            'hespress "2025-09" video twitter',
            'hespress video "sept 2025" twitter',
            'hespress "september 2025" x.com video',
            # Nouvelles requêtes étendues
            'hespress.com video twitter september 2025',
            'hespress maroc video twitter 2025',
            'hespress actualités video septembre 2025',
            'hespress news video twitter 2025',
            '"hespress" "septembre" "2025" "video"',
            'hespress clip video twitter 2025',
            'hespress reportage video septembre'
        ]
        
        all_video_links = []
        
        for i, query in enumerate(queries, 1):
            logging.info(f"🎯 Recherche {i}/{len(queries)}: {query}")
            links = search_google_videos_2025(driver, query, max_results=150)  # Augmenté à 150
            all_video_links.extend(links)
            
            # Sauvegarde intermédiaire des URLs
            with open("hespress_urls_temporaire.txt", "w", encoding="utf-8") as f:
                for url in list(set(all_video_links)):
                    f.write(url + "\n")
            
            logging.info(f"📊 Progression: {len(set(all_video_links))} liens uniques accumulés")
            time.sleep(2)  # Pause courte entre les recherches
        
        # Supprimer les doublons
        unique_links = list(set(all_video_links))
        logging.info(f"🎉 TOTAL LIENS UNIQUES TROUVÉS: {len(unique_links)}")
        
        if not unique_links:
            logging.warning("❌ Aucun lien trouvé pour septembre 2025.")
            return
        
        # Sauvegarder la liste complète des URLs
        with open("hespress_video_urls_septembre_2025_COMPLET.txt", "w", encoding="utf-8") as f:
            for url in unique_links:
                f.write(url + "\n")
        
        # FILTRAGE SEPTEMBRE 2025 - Version optimisée
        september_2025_results = []
        total_a_analyser = len(unique_links)
        
        logging.info(f"🔍 Début de l'analyse et filtrage septembre 2025 sur {total_a_analyser} liens...")
        
        for i, link in enumerate(unique_links, 1):
            try:
                if i % 10 == 0:
                    logging.info(f"📊 Analyse {i}/{total_a_analyser} - {len(september_2025_results)} vidéos septembre 2025 trouvées")
                
                driver.get(link)
                time.sleep(2)  # Réduit le temps d'attente
                
                # Extraction RAPIDE des données
                tweet_data = extract_tweet_data_quick(driver)
                
                # FILTRE SEPTEMBRE 2025
                timestamp = tweet_data.get("timestamp", "")
                if "2025-09" in timestamp:
                    video_data = {
                        "tweet_url": link,
                        "title": tweet_data["title"],
                        "timestamp": timestamp,
                        "author": tweet_data["author"],
                        "metrics": tweet_data["metrics"],
                        "success": True,
                        "scraped_at": datetime.now().isoformat()
                    }
                    september_2025_results.append(video_data)
                    logging.info(f"✅ SEPTEMBRE 2025: {timestamp} - {tweet_data['title'][:100]}...")
                
                # Sauvegarde incrémentale
                if i % 20 == 0:
                    with open("hespress_septembre_2025_temp.json", "w", encoding="utf-8") as f:
                        json.dump(september_2025_results, f, ensure_ascii=False, indent=2)
                
            except Exception as e:
                logging.debug(f"❌ Erreur sur {link}: {e}")
                continue
        
        # SAUVEGARDE FINALE
        save_results_max(september_2025_results, len(unique_links))
        
        # STATISTIQUES DÉTAILLÉES
        logging.info(f"📊 RAPPORT FINAL SEPTEMBRE 2025:")
        logging.info(f"   🔗 Total liens analysés: {len(unique_links)}")
        logging.info(f"   ✅ Vidéos septembre 2025 validées: {len(september_2025_results)}")
        logging.info(f"   📈 Taux de réussite: {len(september_2025_results)/len(unique_links)*100:.1f}%")
        
        if september_2025_results:
            dates = [result["timestamp"] for result in september_2025_results if result.get("timestamp")]
            if dates:
                logging.info(f"   📅 Plage de dates: {min(dates)} à {max(dates)}")
        
    except Exception as e:
        logging.error(f"❌ Erreur générale: {e}")
    finally:
        driver.quit()
        logging.info("🔒 Navigateur fermé")

def save_results_max(results, total_analyses):
    """Sauvegarde les résultats avec statistiques détaillées"""
    try:
        # JSON complet avec métadonnées
        output_data = {
            "metadata": {
                "scraping_date": datetime.now().isoformat(),
                "total_urls_analyzed": total_analyses,
                "september_2025_videos_found": len(results),
                "success_rate": f"{(len(results)/total_analyses*100):.1f}%" if total_analyses > 0 else "0%"
            },
            "videos": results
        }
        
        with open("hespress_videos_septembre_2025_FINAL.json", "w", encoding="utf-8") as f:
            json.dump(output_data, f, ensure_ascii=False, indent=2)
        
        # CSV détaillé
        if results:
            csv_data = []
            for result in results:
                csv_data.append({
                    "tweet_url": result.get("tweet_url", ""),
                    "title": result.get("title", ""),
                    "author": result.get("author", ""),
                    "timestamp": result.get("timestamp", ""),
                    "likes": result.get("metrics", {}).get("likes", 0),
                    "retweets": result.get("metrics", {}).get("retweets", 0),
                    "scraped_at": result.get("scraped_at", "")
                })
            
            df = pd.DataFrame(csv_data)
            df.to_csv("hespress_videos_septembre_2025_FINAL.csv", index=False, encoding="utf-8-sig")
        
        # Fichier texte simple avec URLs
        with open("hespress_urls_septembre_2025_FINAL.txt", "w", encoding="utf-8") as f:
            for result in results:
                f.write(result.get("tweet_url", "") + "\n")
        
        logging.info("💾 FICHIERS FINAUX SAUVEGARDÉS:")
        logging.info("   - hespress_videos_septembre_2025_FINAL.json")
        logging.info("   - hespress_videos_septembre_2025_FINAL.csv")
        logging.info("   - hespress_urls_septembre_2025_FINAL.txt")
        logging.info("   - hespress_video_urls_septembre_2025_COMPLET.txt (tous les liens)")
        
    except Exception as e:
        logging.error(f"❌ Erreur sauvegarde: {e}")

# --------------------------
# VÉRIFICATION RAPIDE
# --------------------------
def check_results_quick():
    """Vérification rapide des résultats"""
    try:
        files = {
            "Fichier JSON complet": "hespress_videos_septembre_2025_FINAL.json",
            "Fichier CSV": "hespress_videos_septembre_2025_FINAL.csv", 
            "URLs septembre 2025": "hespress_urls_septembre_2025_FINAL.txt",
            "Tous les liens trouvés": "hespress_video_urls_septembre_2025_COMPLET.txt"
        }
        
        for nom, fichier in files.items():
            if os.path.exists(fichier):
                taille = os.path.getsize(fichier)
                print(f"✅ {nom}: {fichier} ({taille} octets)")
                
                if fichier.endswith('.json') and taille > 0:
                    with open(fichier, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        if "videos" in data:
                            print(f"   📊 {len(data['videos'])} vidéos septembre 2025")
                        else:
                            print(f"   📊 {len(data)} vidéos septembre 2025")
                elif fichier.endswith('.txt'):
                    with open(fichier, 'r', encoding='utf-8') as f:
                        lignes = f.readlines()
                        print(f"   🔗 {len(lignes)} URLs")
            else:
                print(f"❌ {nom}: {fichier} - NON TROUVÉ")
                
    except Exception as e:
        print(f"❌ Erreur vérification: {e}")

# --------------------------
# EXÉCUTION PRINCIPALE
# --------------------------
if __name__ == "__main__":
    print("=" * 70)
    print("🎥 SCRAPER MAXIMAL HESPRESS SEPTEMBRE 2025")
    print("=" * 70)
    print("⚠️  Ce script va:")
    print("   - Utiliser 16 requêtes Google différentes")
    print("   - Scroller agressivement pour max de résultats") 
    print("   - Analyser jusqu'à 150 liens par requête")
    print("   - Filtrer AUTOMATIQUEMENT septembre 2025")
    print("   - Sauvegarder tous les liens trouvés")
    print("=" * 70)
    
    print("Options:")
    print("1. 🚀 Lancer le scraping MAXIMAL (recommandé)")
    print("2. 🔍 Vérifier les résultats existants")
    
    choix = input("\nVotre choix (1 ou 2): ").strip()
    
    if choix == "1":
        print("🚀 LANCEMENT DU SCRAPING MAXIMAL...")
        print("⏰ Cette opération peut prendre 15-30 minutes.")
        print("📈 Objectif: MAXIMISER le nombre de liens septembre 2025")
        confirmation = input("Confirmez-vous? (o/n): ").strip().lower()
        
        if confirmation == 'o':
            scrape_hespress_videos_september_2025_max()
        else:
            print("❌ Opération annulée.")
    elif choix == "2":
        print("🔍 Vérification des fichiers...")
        check_results_quick()
    else:
        print("❌ Choix invalide. Veuillez choisir 1 ou 2.")

🎥 SCRAPER MAXIMAL HESPRESS SEPTEMBRE 2025
⚠️  Ce script va:
   - Utiliser 16 requêtes Google différentes
   - Scroller agressivement pour max de résultats
   - Analyser jusqu'à 150 liens par requête
   - Filtrer AUTOMATIQUEMENT septembre 2025
   - Sauvegarder tous les liens trouvés
Options:
1. 🚀 Lancer le scraping MAXIMAL (recommandé)
2. 🔍 Vérifier les résultats existants



Votre choix (1 ou 2):  1


🚀 LANCEMENT DU SCRAPING MAXIMAL...
⏰ Cette opération peut prendre 15-30 minutes.
📈 Objectif: MAXIMISER le nombre de liens septembre 2025


Confirmez-vous? (o/n):  o


2025-10-25 19:10:08,141 - INFO - ====== WebDriver manager ======
2025-10-25 19:10:13,098 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-25 19:10:13,253 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-25 19:10:13,402 - INFO - Driver [C:\Users\dell\.wdm\drivers\chromedriver\win64\141.0.7390.122\chromedriver-win32/chromedriver.exe] found in cache
2025-10-25 19:10:14,639 - INFO - 🎯 Recherche 1/17: hespress video twitter september 2025
2025-10-25 19:10:14,641 - INFO - 🔍 Recherche Google 2025: hespress video twitter september 2025
2025-10-25 19:10:20,289 - INFO - 📜 Scroll agressif pour charger plus de résultats...
2025-10-25 19:10:47,883 - INFO - ✅ Lien trouvé: https://x.com/hespress/status/1973155430911357254
2025-10-25 19:11:38,916 - INFO - ✅ Lien trouvé (méthode 3): https://x.com/hespress/status/1973170561812668582
2025-10-25 19:11:38,980 - INFO - ✅ Lien trouvé (méthode 3): https://x.com/hespress/status/1966849350472266010
2025-10-25 19:11:39,025 - I